In [1]:
import datetime as dt
import polars as pl
from trading_engine.core import (
    read_data, create_model_state, orchestrate_model_backtests,
    orchestrate_model_simulations, orchestrate_portfolio_backtests,
    orchestrate_portfolio_simulations
)
from trading_engine.models import MODELS
from trading_engine.optimizers import OPTIMIZERS

from trading_engine.model_state.registry import momentum
from common.constants import ProcessingMode

import datetime

In [5]:
# 1) experiment config
universe = [
  "SPY-US", "SLV-US", "GLD-US", "TLT-US", "USO-US", "UNG-US", "IXJ-US",
  "KXI-US", "JXI-US", "IXG-US", "IXN-US", "RXI-US", "MXI-US", "EXI-US",
  "IXC-US", "IEI-US", "SHY-US", "BIL-US", "JPXN-US", "INDA-US", "MCHI-US",
  "EZU-US", "IBIT-US", "ETHA-US", "VIXY-US"
]
features = ["close_momentum_10", ]                 # must exist in FEATURES
models   = ["RXI_TLT_pml_10", "GLD_USO_nml_10"]    # keys in MODELS
optimizers   = ["equal_weight"]                    # keys in OPTIMIZERS
start, end = datetime.date(2023, 1, 1), datetime.date(2025, 1, 1)

In [7]:
# 2) build model state + prices (cached locally)
lf = read_data()
def build(): 
    ms, prices = create_model_state(lf, features, start, end, universe)
    return ms  # prices are set globally by create_model_state
    
model_state = build()

In [8]:
# 3) run model backtests + simulations
model_insights = orchestrate_model_backtests(models, universe)
model_sims     = orchestrate_model_simulations(model_insights, initial_value=1_000_000)

In [9]:
# 4) optimize portfolio and simulate
portfolio_w    = orchestrate_portfolio_backtests(optimizers, model_insights, model_sims, universe)
portfolio_sims = orchestrate_portfolio_simulations(portfolio_w, initial_value=1_000_000)

In [10]:
# 5) visualize one result (example: equal_weight)
portfolio_sims["equal_weight"]["backtest_metrics"]

metric,value
str,f64
"""total_return""",0.302023
"""annualized_return""",0.141662
"""annualized_volatility""",0.081922
"""sharpe_ratio""",1.72923
"""sortino_ratio""",1.401104
…,…
"""num_weight_events""",502.0
"""parsing_time_ms""",3.0
"""simulation_time_ms""",2.0
